# 02 · Model selection with nested cross-validation

Four model families are compared for each of the five functional attributes:
linear regression, Lasso, k-nearest neighbours (kNN) and support vector regression (SVR).
Each pipeline imputes missing values with the training-fold mean, standardises the
features, tunes hyperparameters with a randomised search and selects features with
sequential floating forward selection (SFFS).

**Why nested CV?** In the original thesis the hyperparameter search and the feature
selection used the same folds that produced the reported scores. Here an outer 5-fold
loop only scores, and all tuning and selection happen in an inner 5-fold loop on the
outer training data. The last section runs both protocols on the same data to show the gap.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
from smalldata import FEATURES, TARGETS, load_dataset

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Set the DATA_PATH environment variable to run on another CSV with the same columns
DATA_PATH = Path(os.environ.get("DATA_PATH", ROOT / "data" / "midsize_sedans.csv"))
ds = load_dataset(DATA_PATH)
print(f"{ds.n_samples} vehicles, {len(FEATURES)} design variables, {len(TARGETS)} functional attributes")

from smalldata import default_candidates, flat_cv, nested_cv, results_table, selection_frequency

N_ITER = 20        # randomised-search draws per inner loop (the thesis used 50)
SEED = 42
candidates = default_candidates(ds.n_samples)

## Nested cross-validation for every target and model

This takes a few minutes on a laptop: 4 models × 5 targets × 5 outer folds, each with its own inner search and SFFS run.

In [ ]:
nested = {}
for target in TARGETS:
    X, y = ds.target(target)
    for cand in candidates:
        nested[(target, cand.name)] = nested_cv(cand, X, y, n_iter=N_ITER, seed=SEED)

table = results_table(list(nested.values()))
table.round(3)

### Out-of-fold R² by target and model

In [ ]:
r2 = table["R2"].unstack("model")[[c.name for c in candidates]].loc[TARGETS]

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(r2.values, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(r2.shape[1]), r2.columns)
ax.set_yticks(range(r2.shape[0]), r2.index)
for i in range(r2.shape[0]):
    for j in range(r2.shape[1]):
        ax.text(j, i, f"{r2.iat[i, j]:.2f}", ha="center", va="center",
                color="white" if r2.iat[i, j] < 0.5 else "black")
fig.colorbar(im, ax=ax, label="R² (out of fold)")
ax.set_title("Nested CV, out-of-fold R²")
fig.tight_layout()

best = r2.idxmax(axis=1).rename("best model").to_frame()
best["R2"] = r2.max(axis=1)
best.round(3)

### Predicted vs actual for the best model per target

Each point is predicted by a model that never saw it during fitting, tuning or feature selection.

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 3.8))
for ax, target in zip(axes, TARGETS):
    res = nested[(target, best.at[target, "best model"])]
    _, y = ds.target(target)
    ax.scatter(y, res.predictions, s=18)
    lims = [min(y.min(), res.predictions.min()), max(y.max(), res.predictions.max())]
    ax.plot(lims, lims, color="black", lw=1)
    ax.set_xlabel(f"actual {target}")
    ax.set_ylabel("predicted")
    ax.set_title(f"{res.model}, R² = {res.metrics['R2']:.2f}")
fig.tight_layout()

### How stable is the feature selection?

Share of the 5 outer folds in which SFFS kept each design variable, for the best model
per target. Values near 0 or 1 mean a stable choice; values around 0.5 mean the selected
subset depends on which handful of vehicles happened to be in the training fold.

In [ ]:
freq = pd.DataFrame({t: selection_frequency(nested[(t, best.at[t, "best model"])], FEATURES) for t in TARGETS})
freq.round(1)

## Original protocol vs nested CV

`flat_cv` reproduces the thesis protocol: tune and select features on all rows with
5-fold CV, then report 5-fold CV scores on the same folds. The difference to the nested
score is the optimism caused by reusing the scoring folds for model selection.

In [ ]:
rows = []
for target in TARGETS:
    X, y = ds.target(target)
    for cand in candidates:
        flat = flat_cv(cand, X, y, n_iter=N_ITER, seed=SEED)
        rows.append({"target": target, "model": cand.name,
                     "R2 flat (thesis protocol)": flat.metrics["R2"],
                     "R2 nested": nested[(target, cand.name)].metrics["R2"]})
compare = pd.DataFrame(rows).set_index(["target", "model"])
compare["optimism"] = compare["R2 flat (thesis protocol)"] - compare["R2 nested"]
compare.round(3)

In [ ]:
summary = compare.groupby("model")[["R2 flat (thesis protocol)", "R2 nested", "optimism"]].median()
ax = summary[["R2 flat (thesis protocol)", "R2 nested"]].plot.bar(figsize=(7, 3.8), rot=0)
ax.set_ylabel("median R² over targets")
ax.set_title("Reusing scoring folds for tuning inflates R²")
plt.tight_layout()
summary.round(3)

## Takeaways

* Reusing the scoring folds for tuning and feature selection inflated R² for every model
  on every target. Averaged over targets (median), the flat protocol scored 0.59–0.62
  for all four models, while nested CV gives 0.26–0.38.
* Geometry-driven attributes are learnable from this data: passenger and cargo volume
  reach R² ≈ 0.65 with Lasso. Performance attributes are much harder (R² ≈ 0.35–0.4
  for fuel efficiency and acceleration), and towing capacity is not predictable from
  these design variables. It only takes a few discrete rating values (1,000–2,000 lb).
* No model family wins everywhere. Lasso and kNN are the most robust; differences of a
  few hundredths in R² are within fold-to-fold noise at n = 31.
* A wide search space can backfire on tiny folds. SVR on fuel efficiency scored 0.58
  under the flat protocol and below zero under nested CV.
* SFFS choices that are physically plausible are also the stable ones: height and
  fuel-tank capacity for both volumes, engine displacement for fuel efficiency and
  acceleration. Other picks change from fold to fold, so report them as frequencies.